# 🩺 第二十六天 · CBLUE 第一个机器学习模型（KUAKE-QIC）

**今天目标（约 1.5 小时）**：用 **TF-IDF + 逻辑回归**跑通 KUAKE-QIC（医疗查询意图分类，11 类），在验证集上拿到你的第一个「机器学习准确率」。

**预期结果（助手已提前验证）**：dev 准确率 **75.7%**——比「全猜最多类」的 34.6% 高出一倍多。

> ⚠️ sklearn 已装好（base 环境，清华镜像）。先 **Kernel → Restart Kernel**。

## ⚠️ 先记住这堂课：中文分词的坑（今天最重要的知识）

如果你直接用默认的 `TfidfVectorizer()`，得到的是 **36.2%**（几乎等于瞎猜）；改成 `analyzer='char'` 才到 **75.7%**。原因：

- 默认分词器按**英文空格/标点**切词，而中文句子没有空格，整句被当成一个"词"，特征全废；
- 中文要靠**字符 n-gram**（`analyzer='char'`）或专业分词库（jieba）。

**这一步是所有中文 NLP 的地基**——你以后做任何中文医疗文本任务都会用到。

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

train = pd.read_json("KUAKE-QIC/KUAKE-QIC_train.json")
dev   = pd.read_json("KUAKE-QIC/KUAKE-QIC_dev.json")
test  = pd.read_json("KUAKE-QIC/KUAKE-QIC_test.json")
print(train.shape, dev.shape, test.shape)

## 第 1 步 · TF-IDF 向量化 + 逻辑回归（跑这个 cell）

- `analyzer='char'`：按**字**切（中文关键一步）；`ngram_range=(1,2)`：同时看单字和双字组合（"血糖""血压"这种词就靠双字捕获）；
- 逻辑回归 = 多分类的"最小可用机器学习模型"，`max_iter` 调大避免收敛警告。

In [ ]:
vec = TfidfVectorizer(analyzer="char", ngram_range=(1, 2), min_df=2)
X_train = vec.fit_transform(train["query"])   # 训练集：fit + transform
X_dev   = vec.transform(dev["query"])         # 验证集：只 transform（不能 fit！）

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, train["label"])

pred = clf.predict(X_dev)
print("dev 准确率: %.4f" % accuracy_score(dev["label"], pred))
print(classification_report(dev["label"], pred, digits=3, zero_division=0))

## 第 2 步 · 读分类报告，做错误分析（写进周报）

看 `classification_report` 的 **f1-score** 一列：
- **强类**（>0.8）：医疗费用、治疗方案、就医建议、病情诊断——特征词明显（"多少钱""怎么办""是什么病"）；
- **弱类**（<0.5）：**指标解读（0.27）、后果表述（0.44）**——训练样本最少（137/235 条）且语义模糊，模型几乎找不出来。

**你的观察**（写在这里）：
1. 哪个类最弱？为什么？
2. 如果想提升，你第一反应会改什么？（提示：样本量 / 特征 / 模型）

In [ ]:
# 第 3 步 · 预测测试集，生成提交文件
test = test.copy()
test["label"] = clf.predict(vec.transform(test["query"]))
submission = test[["id", "query", "label"]]
submission.to_json("KUAKE-QIC/KUAKE-QIC_test_pred.json", orient="records", force_ascii=False)
print("提交文件已生成，共", len(submission), "条")
print(submission.head(3))

## ✅ D26 完成标准（打钩）

- [ ] dev 准确率跑到 **75% 以上**
- [ ] 分类报告看懂：能说出最强的 2 个类和最弱的 2 个类
- [ ] 错误分析写了你的观察
- [ ] 提交文件 `KUAKE-QIC_test_pred.json` 生成成功
- [ ] 保存（**Cmd + S**）

> 完成后喊我验收。**提交说明**：KUAKE-QIC 是 CBLUE 经典任务、榜单已静态，我们以「本地跑通 + 75.7% 基线」作为简历/GitHub 记录；想要活跃榜单就等年底 CCKS/CHIP 2026 报名。

**D27 预告**：简历终稿 + 润色 GitHub README（把「CBLUE 75.7% 基线」这条补进简历和仓库首页）。